In [1]:
import _referAsMain

import math
import time
import random
import sys
import numpy
from IPython.display import SVG, display
from holo.pointers import Pointer
from holo.profilers import Profiler
from holo.prettyFormats import prettyPrint, prettyTime

print(sys.version_info)

import torch
from torch.utils.data import DataLoader

from paths_cfg import TOKENIZER_SAVE_DIRECTORY, OUR_DATASET_DIRECTORY
from dataset import svg_dataset
import tokenizer_pfe.tokenizer_project as tokenizerLib
import metrics.metrics

from LLM.model import Model, GenerationStats
import LLM.nanochat.gpt as nanoChatModel
from LLM.nanochat.common import compute_init, autodetect_device_type

added '/home/holo/dev/PFE_LLM_art_generation' to import paths
sys.version_info(major=3, minor=13, micro=12, releaselevel='final', serial=0)


In [2]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Torch version: 2.9.1+cu128
CUDA available: True
CUDA version: 12.8
Device count: 1


In [45]:
try:
    torch.cuda.empty_cache()
    del model  # type: ignore
    torch.cuda.empty_cache()
except Exception:
    pass
model = Model.load("model_20.5M", versionID=2, device=torch.device("cuda:0"))
model.show_infos()

loading the tokenizer from: /home/holo/dev/PFE_LLM_art_generation/models_save/model_20.5M/tokenizer.json
loading the historique from: /home/holo/dev/PFE_LLM_art_generation/models_save/model_20.5M/versions/version_0002_checkpoint-2/history.json
Padding vocab_size from 585 to 640 for efficiency
Scaling the LR for the AdamW parameters ∝1/√(512/768) = 1.224745
GPTConfig(sequence_len=8192, vocab_size=585, n_layer=6, n_head=1, n_kv_head=1, n_embd=512, window_pattern='SSSL')
20_512_876 total params (embeding: 1_310_720 | last layer: 327_680 | transformer: 18_874_464)
on device: device(type='cuda', index=0), with effective context_size: 4096
-> 316.539 MFLOPs/token 
-> 1.297 TFLOPs with full context


In [49]:
print(f"trained for {model.nb_epoches_done} epoches")
for k, v in model.historique.get_all_historique().items():
    print(f"  {k}: {v.get(model.nb_epoches_done-1, None):.4g}")  # type: ignore

trained for 2 epoches
  CE_train: 0.787
  CE2_train: 0.672
  PPL_train: 2.197
  PPL2_train: 1.958
  BPC_train: 0.5752
  ENTROPY_train: 0.7843
  LOGITS_SD_train: 3.875
  TOP-1_train: 0.8072
  TOP-5_train: 0.8589
  epoch_duration: 1092
